# Lumbar Spine L4 Extraction on AP Views — BUU-LSPINE + YOLOv8

Instead of using SpineFM (which is lateral-view specific), this pipeline is tailored specifically for the Anteroposterior (AP) lumbar spine domain.

Pipeline Flow:
1. Label Conversion: Convert BUU-LSPINE AP labels (L1–L5 corner coordinates) into the standard YOLO format.
2. Model Training: Train a YOLOv8 object detection model using 5 distinct classes (L1 to L5).
3. Inference & Extraction: Apply the trained model to your custom AP DICOM images $\rightarrow$ Identify the 4th vertebra from the top (L4) $\rightarrow$ Extract (Bounding Box + Crop + Overlay).

Dataset Links:
 - Official Page: https://services.informatics.buu.ac.th/spine/#sq-tab1
 - GitHub Repository: https://github.com/North-Github/BUU-LSPINE (AP+LA, 7,200 images, L1–L5 labels)


## 0. 설치

In [1]:
# !pip install ultralytics pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg opencv-python-headless
import os, glob, re, random, shutil
import numpy as np
from PIL import Image
import cv2
print("ready")

ready


## 1. 경로 설정
BUU-LSPINE 를 내려받아 압축을 푼 뒤, AP 이미지 폴더와 AP 라벨 폴더 경로를 지정하세요.

In [2]:
# ---- BUU-LSPINE (학습 데이터) ----
BUU_AP_IMAGES = "./BUU-LSPINE/AP/images"   # AP X-ray 이미지 폴더 (.jpg/.png)
BUU_AP_LABELS = "./BUU-LSPINE/AP/labels"   # AP 라벨(.txt) 폴더 (이미지와 같은 stem)

# ---- YOLO 학습 셋 출력 위치 ----
YOLO_ROOT = "./yolo_ap_lspine"
VAL_RATIO = 0.15
SEED = 42

# 클래스: 위->아래 순서로 L1..L5 (인덱스 0..4). L4 = 인덱스 3.
CLASS_NAMES = ["L1", "L2", "L3", "L4", "L5"]
L4_CLASS_ID = CLASS_NAMES.index("L4")

# 한 척추체를 이루는 모서리 '엣지 라인' 수 (상/하 엔드플레이트 = 2)
LINES_PER_VERTEBRA = 2
N_VERTEBRAE = 5
random.seed(SEED)
print("L4 class id =", L4_CLASS_ID)

L4 class id = 3


## 2. 라벨 형식 점검 (먼저 실행!)
실제 BUU-LSPINE 라벨 파일 한 개를 그대로 출력합니다. 아래 파서의 가정과 맞는지 눈으로 확인하세요.
- 가정: 한 파일에 **10줄(엣지 라인)**, 각 줄에 숫자 4개 = `(x_left, y_left, x_right, y_right)`.
- 연속한 2줄 = 한 척추체(상·하 엔드플레이트). 총 5개 척추체.

In [3]:
lbls = sorted(glob.glob(os.path.join(BUU_AP_LABELS, "*")))
print("라벨 파일 수:", len(lbls))
if lbls:
    print("예시:", lbls[0], "\n----- 원본 내용 -----")
    with open(lbls[0]) as f:
        print(f.read())

라벨 파일 수: 400
예시: ./BUU-LSPINE/AP/labels\0001-F-037Y0.csv 
----- 원본 내용 -----
876.2222,167.0618,1111.472,168.7665,0
865.9939,313.6671,1119.996,313.6671,0
870.8508,344.4474,1116.587,351.1707,0
847.6437,498.0999,1119.997,511.2078,0
842.3262,553.4754,1108.814,557.2025,0
825.0808,705.7509,1118.291,724.5027,0
826.1249,774.749,1111.775,787.0792,0
801.4645,937.0969,1113.83,957.6473,0
797.71,983.067,1115.284,980.1932,0
766.5613,1163.979,1129.421,1157.696,0



## 3. 라벨 파서 + YOLO 변환
각 척추체의 모서리 4점 → 바운딩박스(min/max)로 변환합니다.
**클래스는 파일 순서를 믿지 않고, 5개 척추체를 y(세로)로 정렬해 위→아래 L1..L5 로 부여**합니다
(파일이 위→아래든 아래→위든 무관하게 안전).

In [4]:
def _read_floats_per_line(path):
    rows = []
    with open(path) as f:
        for line in f:
            nums = re.findall(r"-?\d+\.?\d*", line)
            if len(nums) >= 4:
                rows.append([float(v) for v in nums[:4]])  # xL,yL,xR,yR
    return rows  # 기대: 10줄

def parse_annotation(path):
    """라벨 파일 -> L1..L5 순서의 bbox 리스트(픽셀). 실패 시 []"""
    rows = _read_floats_per_line(path)
    need = LINES_PER_VERTEBRA * N_VERTEBRAE
    if len(rows) < need:
        return []
    verts = []
    for i in range(N_VERTEBRAE):
        pts = []
        for j in range(LINES_PER_VERTEBRA):
            xL, yL, xR, yR = rows[i*LINES_PER_VERTEBRA + j]
            pts += [(xL, yL), (xR, yR)]
        xs = [p[0] for p in pts]; ys = [p[1] for p in pts]
        x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
        verts.append((y1, (x1, y1, x2, y2)))     # y1 로 정렬용
    verts.sort(key=lambda v: v[0])                # 위->아래
    return [b for _, b in verts]                  # L1..L5 순서의 bbox 리스트

def to_yolo_line(cls_id, box, W, H):
    x1, y1, x2, y2 = box
    cx = (x1 + x2) / 2 / W; cy = (y1 + y2) / 2 / H
    w = (x2 - x1) / W;      h = (y2 - y1) / H
    return f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"

def find_image(stem):
    for ext in (".jpg", ".jpeg", ".png", ".JPG", ".PNG"):
        p = os.path.join(BUU_AP_IMAGES, stem + ext)
        if os.path.exists(p):
            return p
    return None

# ---- 변환 실행 ----
for sub in ["images/train","images/val","labels/train","labels/val"]:
    os.makedirs(os.path.join(YOLO_ROOT, sub), exist_ok=True)

label_files = sorted(glob.glob(os.path.join(BUU_AP_LABELS, "*")))
random.shuffle(label_files)
n_val = int(len(label_files) * VAL_RATIO)
ok = skip = 0
for k, lp in enumerate(label_files):
    stem = os.path.splitext(os.path.basename(lp))[0]
    ip = find_image(stem)
    if ip is None:
        skip += 1; continue
    boxes = parse_annotation(lp)
    if len(boxes) != N_VERTEBRAE:
        skip += 1; continue
    im = cv2.imread(ip); H, W = im.shape[:2]
    split = "val" if k < n_val else "train"
    shutil.copy(ip, os.path.join(YOLO_ROOT, "images", split, os.path.basename(ip)))
    with open(os.path.join(YOLO_ROOT, "labels", split, stem + ".txt"), "w") as f:
        for cid, box in enumerate(boxes):           # cid 0..4 = L1..L5
            f.write(to_yolo_line(cid, box, W, H) + "\n")
    ok += 1
print(f"변환 완료: {ok}장, 건너뜀 {skip}장 (val {n_val})")

변환 완료: 400장, 건너뜀 0장 (val 60)


## 4. data.yaml 생성

In [5]:
yaml_path = os.path.join(YOLO_ROOT, "data.yaml")
with open(yaml_path, "w") as f:
    f.write(f"path: {os.path.abspath(YOLO_ROOT)}\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write(f"nc: {len(CLASS_NAMES)}\n")
    f.write(f"names: {CLASS_NAMES}\n")
print(open(yaml_path).read())

path: c:\Users\csm02\Desktop\edward\bmd\1 src\yolo_ap_lspine
train: images/train
val: images/val
nc: 5
names: ['L1', 'L2', 'L3', 'L4', 'L5']



## 5. YOLOv8 학습
GPU 권장. epochs/imgsz 는 데이터·자원에 맞게 조정하세요.

In [6]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")          # 더 정확히: yolov8s.pt / yolov8m.pt
model.train(data=yaml_path, epochs=100, imgsz=640, batch=16,
            project="ap_lspine", name="l1_l5", patience=20, workers=2)
BEST = "ap_lspine/l1_l5/weights/best.pt"
print("best weights ->", BEST)

New https://pypi.org/project/ultralytics/8.4.67 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.60  Python-3.12.13 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./yolo_ap_lspine\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mo

## 6. 내 AP DICOM 에 추론 → L4 추출
DICOM 을 BUU-LSPINE 와 비슷한 명암(보통 뼈가 밝은 일반 방사선영상)으로 변환합니다.
`INVERT` 로 명암을 맞추세요(학습 이미지와 같은 톤이 되도록).

In [9]:
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut

DICOM_DIR  = "./dataset-dcm/train/gather/"
OUT_DIR    = "./l4_output_ap"
BEST       = "./runs/detect/ap_lspine/l1_l5/weights/best.pt"
INVERT     = True       # BUU-LSPINE 가 뼈-밝은 톤이면 MONOCHROME1 반전. 2번처럼 미리보기로 맞추세요.
CONF       = 0.25

os.makedirs(OUT_DIR, exist_ok=True)
det = YOLO(BEST)

def dicom_to_rgb(path, invert=INVERT):
    ds = pydicom.dcmread(path)
    arr = ds.pixel_array.astype(np.float32)
    arr = arr*float(getattr(ds,"RescaleSlope",1.0)) + float(getattr(ds,"RescaleIntercept",0.0))
    try: arr = apply_voi_lut(arr.astype(ds.pixel_array.dtype), ds).astype(np.float32)
    except Exception: pass
    if invert and str(getattr(ds,"PhotometricInterpretation","")).upper()=="MONOCHROME1":
        arr = arr.max()-arr
    lo,hi = np.percentile(arr,0.5), np.percentile(arr,99.5)
    if hi<=lo: lo,hi = arr.min(), arr.max()+1e-6
    arr = np.clip((arr-lo)/(hi-lo),0,1)
    return cv2.cvtColor((arr*255).astype(np.uint8), cv2.COLOR_GRAY2RGB)

def pick_l4(boxes, classes, confs):
    """1순위: L4 클래스 박스. 없으면 위->아래 정렬 4번째."""
    if len(boxes)==0: return None, "검출 없음"
    idx = [i for i,c in enumerate(classes) if c==L4_CLASS_ID]
    if idx:
        b = max(idx, key=lambda i: confs[i])
        return boxes[b], "L4 클래스 직접"
    order = sorted(range(len(boxes)), key=lambda i: (boxes[i][1]+boxes[i][3])/2)  # y center
    if len(order)>=4:
        return boxes[order[3]], "위->아래 4번째(폴백)"
    return None, f"척추체 {len(boxes)}개뿐"

ok=fail=0
for path in sorted(glob.glob(os.path.join(DICOM_DIR,"*.dcm"))):
    stem = os.path.splitext(os.path.basename(path))[0]
    try:
        rgb = dicom_to_rgb(path)
        r = det.predict(rgb, conf=CONF, verbose=False)[0]
        boxes = r.boxes.xyxy.cpu().numpy()
        classes = r.boxes.cls.cpu().numpy().astype(int)
        confs = r.boxes.conf.cpu().numpy()
        box, status = pick_l4(boxes, classes, confs)
        if box is None:
            print(f"[skip] {stem[:14]}: {status}"); fail+=1; continue
        x1,y1,x2,y2 = [int(v) for v in box]
        crop = rgb[max(0,y1):y2, max(0,x1):x2]
        cv2.imwrite(os.path.join(OUT_DIR, f"{stem}_L4_crop.png"), cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
        np.save(os.path.join(OUT_DIR, f"{stem}_L4_bbox.npy"), np.array([x1,y1,x2,y2]))
        vis = rgb.copy(); cv2.rectangle(vis,(x1,y1),(x2,y2),(0,255,0),2)
        cv2.putText(vis,"L4",(x1,max(0,y1-5)),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,255,0),2)
        cv2.imwrite(os.path.join(OUT_DIR, f"{stem}_L4_overlay.png"), cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
        print(f"[ ok ] {stem[:14]}: {status} | L4 bbox=({x1},{y1},{x2},{y2})"); ok+=1
    except Exception as e:
        print(f"[fail] {stem[:14]}: {e}"); fail+=1
print(f"\n완료. L4 {ok}건 / 실패 {fail}건 -> {OUT_DIR}")

[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(94,102,130,120)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(101,43,145,63)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(112,109,150,130)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(106,19,137,31)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(101,108,139,126)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(109,135,147,158)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(109,86,150,105)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(114,86,155,103)
[skip] 1.2.826.0.1.36: 척추체 2개뿐
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(98,82,136,98)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(112,123,156,146)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(90,89,142,110)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(102,85,142,103)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(122,103,170,119)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(99,0,154,11)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(108,89,153,110)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(111,20,145,44)
[ ok ] 1.2.82

## 참고
- **픽셀 마스크**가 필요하면 검출(detect) 대신 **YOLOv8-seg**(`yolov8n-seg.pt`)로 학습하세요.
  단, BUU-LSPINE 는 모서리 점 라벨이라 사각형 폴리곤 마스크가 되며, 정밀 분할은 별도 마스크 라벨이 필요합니다.
- 추론 톤(`INVERT`)은 반드시 학습 이미지와 같은 명암으로 맞추세요(2번 점검 셀 활용).
- 검출이 약하면: yolov8s/m 로 키우기, epochs 늘리기, 데이터 증강(mosaic 등) 확인.